# GNN Inference on Multi-Core CPUs and GPUs — Benchmark & Evaluation Suite

Questo notebook esegue la suite completa di benchmarking, verifica e valutazione delle prestazioni come specificato in `requirements.md` e `semantics.md`.

### Requisiti coperti:
- **`BEN-COMP-01..03`**: Confronto tra CPU Sequenziale, Multi-threaded OpenMP e GPU CUDA per **GCN** e **GraphSAGE**.
- **`BEN-COMP-04`**: Caso di scala a **1 milione di nodi** ($N=1.000.000$) e scaling di taglia del grafo.
- **`BEN-COMP-05..07`**: Studio di scalabilità lungo gli altri assi (feature width $F=32, 128$, profondità $D=2, 8$, skew gradi $S=0, 1, 2$).
- **`BEN-COMP-10..11`**: Confronto con framework esterno (**PyTorch Geometric** con `GCNConv` e `SAGEConv` standard) su CPU e CUDA.
- **`VER-01..07`**: Verifica automatica della correttezza numerica con tolleranza floating point (errore $< 10^{-4}$) e test su casi limite.
- **`BEN-MET-01..06`**: Misurazione separata di tempi di compute, memoria picco GPU/host, throughput e speedup.

## 1. Setup Ambiente, GPU e Compilazione

In [ ]:
%%bash
echo '=== GPU Info ==='
nvidia-smi
echo -e '\n=== Toolchain Compilatori ==='
g++ --version | head -n 1
nvcc --version | tail -n 2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
if [ -d "$PROJECT/.git" ]; then
  git -C "$PROJECT" pull --ff-only
else
  git clone https://github.com/Alby02/cuda-gnn-inference.git "$PROJECT"
fi


In [ ]:
%pip install -q meson ninja networkit scipy ogb torch-geometric matplotlib pandas


In [ ]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"

echo '=== Configurazione e Compilazione C++/CUDA (Meson + Ninja) ==='
if [ -d builddir/meson-private ]; then
  meson setup --reconfigure builddir
else
  meson setup builddir
fi
meson compile -C builddir
./builddir/gnn --help


## 2. Smoke Test Rapido (Sanity Check)
Verifica preliminare che i tre backend (`sequential`, `parallel`, `cuda`) producano risultati equivalenti sul demo integrato.

In [ ]:
import subprocess
import re
import numpy as np
from pathlib import Path

project = Path('/content/drive/MyDrive/cuda-gnn-inference')
executable = project / 'builddir' / 'gnn'

def run_mode(mode):
    res = subprocess.run([str(executable), '--backend', mode], capture_output=True, text=True, check=True)
    rows = []
    for line in res.stdout.splitlines():
        m = re.search(r'\[([^\]]+)\]', line)
        if m:
            rows.append([float(x) for x in m.group(1).split(',')])
    return np.array(rows, dtype=np.float32)

seq_out = run_mode('sequential')
par_out = run_mode('parallel')
cuda_out = run_mode('cuda')

np.testing.assert_allclose(par_out, seq_out, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(cuda_out, seq_out, atol=1e-5, rtol=1e-5)
print('OK: Smoke test completato! Sequenziale, OpenMP e CUDA coincidono perfettamente:', cuda_out.tolist())


## 3. Validazione Correttezza su Casi Limite (`VER-03..05`)
Verifica su grafi vuoti (0 archi), nodi isolati, grafo non orientato e layer senza bias rispetto al reference matematico.

In [ ]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"
python synth_data/framework-validation/check_edge_cases.py ./builddir/gnn


## 4. Benchmark Multilayer: Nativo (CPU/CUDA) vs. PyTorch Geometric (`BEN-COMP-10..11`)
Confronto a 3 layer per **GCN**, **GraphSAGE** e modello **Misto (GCN + GraphSAGE + GCN)**.
Valuta la correttezza numerica e misura lo speedup rispetto a PyTorch Geometric (CPU e CUDA).

In [ ]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"
OUT_DIR="$PROJECT/colab_multilayer_benchmark"
mkdir -p "$OUT_DIR"

python scripts/compare_framework.py \
  --native ./builddir/gnn \
  --graph synth_data/forward-validation/multilayer/unweighted.bin_graph \
  --features synth_data/forward-validation/multilayer/features.bin_matrix \
  --model synth_data/forward-validation/multilayer/gcn.model \
          synth_data/forward-validation/multilayer/graphsage.model \
          synth_data/forward-validation/multilayer/mixed.model \
  --backend sequential parallel cuda \
  --threads 1 2 4 \
  --block-size 128 256 \
  --warmups 2 \
  --repetitions 10 \
  --output-dir "$OUT_DIR"


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path('/content/drive/MyDrive/cuda-gnn-inference/colab_multilayer_benchmark/comparison.csv')
if csv_path.exists():
    df = pd.read_csv(csv_path)
    cols = ['model_types', 'native_backend', 'framework_device', 'threads', 'block_size',
            'verification', 'max_abs_error', 'native_mean_ms', 'framework_mean_ms', 'native_speedup']
    display(df[cols].sort_values(by=['model_types', 'native_backend']))


## 5. Valutazione su Dataset Pubblico Reale (`DATA-PUB-01`)
Esegue il benchmark su dataset reale di riferimento (**Planetoid Cora** o **OGBN-ArXiv**).
Il comando scarica automaticamente il dataset, genera i pesi del modello ed esegue il confronto completo tra tutti i backend.

In [ ]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"
OUT_DIR="$PROJECT/colab_cora_results"

# Esecuzione su dataset Cora (per ogbn-arxiv sostituire con --dataset ogbn-arxiv)
python scripts/run_experiments.py \
  --native ./builddir/gnn \
  --dataset Cora \
  --backend sequential parallel cuda \
  # --threads omesso usa la scala automatica 2^x fino al max CPU core (es. 1 2 su Colab)
  --block-size 128 256 \
  --warmups 2 \
  --repetitions 10 \
  --output-dir "$OUT_DIR"


In [ ]:
import pandas as pd
from pathlib import Path

cora_csv = Path('/content/drive/MyDrive/cuda-gnn-inference/colab_cora_results/comparison.csv')
if cora_csv.exists():
    display(pd.read_csv(cora_csv))


## 6. Studio di Scalabilità Parametrica Multi-Asse (BEN-COMP-04..07)
Esegue lo sweep sperimentale completo variando una dimensione alla volta:
- **Nodi ($)**: 1.000, 10.000 (ordine di grandezza)
- **Feature ($)**: 32, 128 canali
- **Profondità ($)**: 2, 8 layer
- **Skew ($)**: 0 (uniforme), 1, 2 (legge di potenza)
- **Backend**: Sequential, Parallel (OpenMP: scala esponenziale ^x$ automatica fino al max di core CPU) e CUDA GPU (block size: 128 e 256)


In [ ]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"
OUT_DIR="$PROJECT/colab_scaling_results"

python scripts/run_experiments.py \
  --native ./builddir/gnn \
  --dataset none \
  --nodes 1000 10000 \
  --widths 32 128 \
  --depths 2 8 \
  --skews 0 1 2 \
  --backend sequential parallel cuda \
  # --threads omesso: auto-scaling esponenziale 2^x (1, 2 su Colab; 1, 2, 4, 8, 16 su PC)
  --block-size 128 256 \
  --warmups 2 \
  --repetitions 10 \
  --output-dir "$OUT_DIR"


## 7. Large-Scale Benchmark: 1 Milione di Nodi (`BEN-COMP-04`)
Come specificato in `requirements.md` (`BEN-COMP-04`) e `project.md`:
> *"The evaluation shall measure graph-size scalability across at least one order of magnitude and shall include a million-node-scale case when permitted by the available memory..."*

Questa sezione genera un grafo sintetico a scala di **1.000.000 di nodi** (~8.000.000 di archi) ed esegue:
- Benchmark GCN e GraphSAGE su GPU CUDA (block size 128 e 256) e CPU OpenMP (2 e 4 thread)
- Confronto prestazionale con PyTorch Geometric su GPU CUDA
- Calcolo del Throughput ($	ext{nodi/s}$ ed $	ext{archi/s}$) e verifica numerica su 1 milione di nodi

In [ ]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"
OUT_DIR="$PROJECT/colab_million_nodes_results"

# Esecuzione su 1.000.000 di nodi (GPU CUDA e CPU OpenMP)
python scripts/run_experiments.py \
  --native ./builddir/gnn \
  --dataset none \
  --nodes 1000000 \
  --widths 32 \
  --depths 2 \
  --skews 0 \
  --backend parallel cuda \
  # --threads omesso: auto-scaling esponenziale 2^x
  --block-size 128 256 \
  --warmups 1 \
  --repetitions 5 \
  --output-dir "$OUT_DIR"


In [ ]:
import pandas as pd
from pathlib import Path

csv_1m = Path('/content/drive/MyDrive/cuda-gnn-inference/colab_million_nodes_results/comparison.csv')
if csv_1m.exists():
    df_1m = pd.read_csv(csv_1m)
    print(f'=== Risultati Benchmark 1 Milione di Nodi (8M archi) ===')
    cols = ['model_types', 'native_backend', 'framework_device', 'threads', 'block_size',
            'verification', 'max_abs_error', 'native_mean_ms', 'framework_mean_ms', 'native_speedup']
    display(df_1m[cols])
else:
    print('File non trovato in:', csv_1m)


## 8. Visualizzazione Grafici e Report Finale
Mostra direttamente nel notebook i grafici di scaling generati (`plots/`) e le tabelle di sintesi.

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import pandas as pd

plots_dir = Path('/content/drive/MyDrive/cuda-gnn-inference/colab_scaling_results/plots')
if plots_dir.exists():
    pngs = sorted(plots_dir.glob('*-scaling.png')) + sorted(plots_dir.glob('*-speedup.png'))
    for p in pngs:
        print(f'\n========== {p.name} ==========')
        display(Image(filename=str(p)))
else:
    print('Nessun grafico trovato in:', plots_dir)


In [ ]:
scale_csv = Path('/content/drive/MyDrive/cuda-gnn-inference/colab_scaling_results/comparison.csv')
if scale_csv.exists():
    df_scale = pd.read_csv(scale_csv)
    print(f'Riepilogo {len(df_scale)} configurazioni sperimentali confrontate:')
    display(df_scale[['workload', 'model_types', 'native_backend', 'framework_device', 'threads', 'block_size', 'verification', 'native_mean_ms', 'framework_mean_ms', 'native_speedup']])
